In [ ]:
!pip install -q torch transformers peft bitsandbytes sentence-transformers faiss-cpu pandas gradio huggingface_hub accelerate

In [ ]:
import os
import pandas as pd
import torch
import gradio as gr
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from huggingface_hub import hf_hub_download
from sentence_transformers import SentenceTransformer
import faiss
import numpy as np

# --- 1. Configuration ---
HF_REPO_ID = "younginAI/mistral-lol-agent"
BASE_MODEL = "unsloth/mistral-7b-v0.2-bnb-4bit"
CSV_FILENAME = "master_meta.csv"

print("🚀 Starting LoL Expert Agent system on Colab GPU...")

# --- 2. RAG Engine Setup ---
try:
    csv_path = hf_hub_download(repo_id=HF_REPO_ID, filename=CSV_FILENAME)
    df = pd.read_csv(csv_path, encoding='utf-8')
    df['context'] = df.apply(lambda x: f"Champion: {x['champion']} | WR: {x['win_rate']}% | Tier: {x['tier']} | Core: {x['core_items']}", axis=1)
    documents = df['context'].tolist()

    embedder = SentenceTransformer('jhgan/ko-sroberta-multitask')
    doc_embeddings = embedder.encode(documents)
    index = faiss.IndexFlatL2(doc_embeddings.shape[1])
    index.add(np.array(doc_embeddings).astype('float32'))
    print("✅ RAG engine setup complete.")
except Exception as e:
    print(f"❌ RAG initialization failed: {e}")
    documents = ["Database connection error."]
    embedder = SentenceTransformer('jhgan/ko-sroberta-multitask')
    index = faiss.IndexFlatL2(768)
    index.add(np.zeros((1, 768)).astype('float32'))

# --- 3. Model Loading (T4 GPU 4bit Optimization) ---
print("🧠 Loading model and merging LoRA adapters...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16
)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto"
)
model = PeftModel.from_pretrained(base_model, HF_REPO_ID)
model.eval()
print("✅ AI Model loaded successfully.")

# --- 4. Inference Logic ---
def respond(message, history):
    query_vec = embedder.encode([message])
    _, indices = index.search(np.array(query_vec).astype('float32'), k=1)
    context = documents[indices[0][0]]

    prompt = f"""System: Advanced LoL Strategy Agent. Use the provided context and your fine-tuned knowledge.
Context: {context}
User: {message}
Answer:"""

    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = model.generate(**inputs, max_new_tokens=512, temperature=0.7)

    return tokenizer.decode(outputs[0], skip_special_tokens=True).split("Answer:")[-1].strip()

# --- 5. UI Deployment ---
print("🌐 Launching web interface...")
demo = gr.ChatInterface(
    fn=respond,
    title="LoL Strategic Expert Agent",
    description="Answers champion strategies based on real-time MCP win rate data and fine-tuned knowledge.",
    theme="glass"
)

# Launch with public link for external access
demo.launch(share=True)